# Data Science + Big Data + SQL Foundations (Project Learning Notebook)

Goal: after this notebook, you should be able to read this project confidently, design tables, write SQL queries,
and understand the data-science and big-data workflow behind transport analytics.

## Learning map

1. Data science basics with a transport dataset
2. Big data foundations (why architecture matters)
3. SQL fundamentals (all main actions + operators + symbols)
4. Joins, aggregations, subqueries, CTEs, and window functions
5. PostgreSQL-specific syntax used in this project

In [ ]:
from __future__ import annotations

import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

rng = np.random.default_rng(42)
plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
# Build a synthetic transport dataset for teaching
dates = pd.date_range("2024-01-01", periods=90, freq="D")

stations = pd.DataFrame(
    {
        "station_id": [101, 102, 201, 202, 301, 302],
        "station_name": [
            "Montparnasse",
            "Bastille",
            "Chatelet",
            "Nation",
            "Times Sq",
            "14 St",
        ],
        "line_code": ["M4", "M5", "M1", "M1", "A", "L"],
        "city": ["Paris", "Paris", "Paris", "Paris", "New York", "New York"],
        "region": ["Ile-de-France", "Ile-de-France", "Ile-de-France", "Ile-de-France", "NYC", "NYC"],
    }
)

ride_rows = []
ride_id = 1
for d in dates:
    dow = d.dayofweek
    weekend_factor = 0.72 if dow >= 5 else 1.0
    for station_id in stations["station_id"]:
        base = 1300 if station_id < 300 else 1800
        seasonal = 1 + 0.15 * np.sin((d.dayofyear / 365) * 2 * np.pi)
        noise = rng.normal(0, 120)
        riders = max(50, int((base * weekend_factor * seasonal) + noise))
        ride_rows.append((ride_id, station_id, d, riders))
        ride_id += 1

rides = pd.DataFrame(ride_rows, columns=["ride_id", "station_id", "ride_date", "riders"])  

weather_rows = []
for d in dates:
    weather_rows.append((d, "Ile-de-France", round(rng.normal(12, 7), 1), round(max(0, rng.gamma(1.7, 1.8)), 1)))
    weather_rows.append((d, "NYC", round(rng.normal(14, 8), 1), round(max(0, rng.gamma(1.9, 2.1)), 1)))
weather = pd.DataFrame(weather_rows, columns=["weather_date", "region", "mean_temp_c", "precip_mm"])

holidays = pd.DataFrame(
    {
        "holiday_date": pd.to_datetime(["2024-01-01", "2024-01-15", "2024-02-19"]),
        "country_code": ["FR", "US", "US"],
        "holiday_name": ["New Year", "MLK Day", "Presidents Day"],
    }
)

rides.head(), stations, weather.head(), holidays

## Part A - Data Science Basics

Data science workflow:

- Understand data shape and meaning
- Clean and validate data
- Explore patterns with statistics and visualization
- Engineer useful features
- Build and evaluate models

In [ ]:
# Basic descriptive analysis
print("rides shape:", rides.shape)
print("stations shape:", stations.shape)
print("weather shape:", weather.shape)

summary = rides["riders"].describe().to_frame("riders_summary")
summary

In [ ]:
# Merge and inspect daily totals
rides_with_station = rides.merge(stations, on="station_id", how="left")
daily_totals = rides_with_station.groupby(["ride_date", "region"], as_index=False)["riders"].sum()

fig, ax = plt.subplots(figsize=(12, 4))
for region, g in daily_totals.groupby("region"):
    ax.plot(g["ride_date"], g["riders"], label=region)
ax.set_title("Daily riders by region")
ax.set_xlabel("Date")
ax.set_ylabel("Riders")
ax.legend()
plt.show()

daily_totals.head()

In [ ]:
# Distribution + spread charts
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rides["riders"], bins=25, color="#1f77b4")
axes[0].set_title("Histogram of riders")

box_data = [
    rides_with_station.loc[rides_with_station["region"] == "Ile-de-France", "riders"],
    rides_with_station.loc[rides_with_station["region"] == "NYC", "riders"],
]
axes[1].boxplot(box_data, tick_labels=["Ile-de-France", "NYC"])
axes[1].set_title("Boxplot by region")

merged_weather = daily_totals.merge(weather, left_on=["ride_date", "region"], right_on=["weather_date", "region"], how="left")
axes[2].scatter(merged_weather["precip_mm"], merged_weather["riders"], alpha=0.6)
axes[2].set_title("Rain vs riders")
axes[2].set_xlabel("precip_mm")
axes[2].set_ylabel("riders")

plt.tight_layout()
plt.show()

## Part B - Big Data Basics

Big data is not only "large files". It is usually explained with the **5V** model:

- **Volume**: very large amount of data
- **Velocity**: data arrives quickly (streams)
- **Variety**: many formats (CSV, JSON, logs, text, images)
- **Veracity**: uncertain quality/noise
- **Value**: business impact of extracted insights

In this project:
- MTA hourly data introduces **volume + velocity**
- French + US sources introduce **variety**
- Missing/dirty categories introduce **veracity**
- Forecasting demand and anomalies is the **value**

## Part C - SQL Fundamentals (Actions, Operators, Symbols)

### 1) SQL action families

- **DDL** (schema): `CREATE`, `ALTER`, `DROP`, `TRUNCATE`
- **DML** (data): `SELECT`, `INSERT`, `UPDATE`, `DELETE`, `MERGE`
- **DCL** (permissions): `GRANT`, `REVOKE`
- **TCL** (transactions): `BEGIN`, `COMMIT`, `ROLLBACK`, `SAVEPOINT`

### 2) Core SQL symbols and notations

- `*` all columns
- `;` end statement
- `'text'` string literal
- `"columnName"` quoted identifier
- `%` wildcard (many chars in `LIKE`)
- `_` wildcard (single char in `LIKE`)
- `()` grouping / function arguments
- `--` single-line comment
- `/* ... */` multi-line comment

### 3) Important operators/predicates

- Comparisons: `=`, `<>`, `!=`, `>`, `<`, `>=`, `<=`
- Logical: `AND`, `OR`, `NOT`
- Range/set: `BETWEEN`, `IN`, `NOT IN`
- Pattern: `LIKE`, `ILIKE` (PostgreSQL)
- Null checks: `IS NULL`, `IS NOT NULL`
- Existence: `EXISTS`, `NOT EXISTS`
- PostgreSQL JSON examples: `->`, `->>`, `@>`

In [ ]:
# Create an in-memory SQL database for all runnable SQL examples
conn = sqlite3.connect(":memory:")

stations.to_sql("stations", conn, index=False, if_exists="replace")
rides.to_sql("rides", conn, index=False, if_exists="replace")
weather.to_sql("weather", conn, index=False, if_exists="replace")
holidays.to_sql("holidays", conn, index=False, if_exists="replace")

def run_sql(query: str) -> pd.DataFrame:
    return pd.read_sql_query(query, conn)

def exec_sql(query: str) -> None:
    conn.execute(query)
    conn.commit()

run_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")

In [ ]:
# DDL examples
exec_sql("CREATE TABLE ticket_sales (sale_id INTEGER PRIMARY KEY, station_id INTEGER, amount REAL, sale_date TEXT);")
exec_sql("ALTER TABLE ticket_sales ADD COLUMN payment_method TEXT;")

# Check resulting schema
run_sql("PRAGMA table_info(ticket_sales);")

In [ ]:
# DML examples: INSERT, UPDATE, DELETE, SELECT
exec_sql("INSERT INTO ticket_sales (sale_id, station_id, amount, sale_date, payment_method) VALUES (1, 101, 45.5, '2024-01-01', 'card');")
exec_sql("INSERT INTO ticket_sales (sale_id, station_id, amount, sale_date, payment_method) VALUES (2, 201, 17.0, '2024-01-02', 'cash');")

exec_sql("UPDATE ticket_sales SET amount = amount * 1.1 WHERE payment_method = 'cash';")
exec_sql("DELETE FROM ticket_sales WHERE sale_id = 1;")

run_sql("SELECT * FROM ticket_sales;")

In [ ]:
# WHERE + operators + symbols examples
q = """
SELECT
    ride_id,
    station_id,
    riders
FROM rides
WHERE riders >= 1500
  AND station_id IN (301, 302)
  AND ride_date BETWEEN '2024-01-01' AND '2024-02-15'
ORDER BY riders DESC
LIMIT 10;
"""
run_sql(q)

## Joins (syntax + meaning)

- **INNER JOIN**: only matching rows in both tables
- **LEFT JOIN**: all rows from left table + matches from right table
- **RIGHT JOIN**: all rows from right table + matches from left table (supported in PostgreSQL)
- **FULL OUTER JOIN**: all rows from both tables (supported in PostgreSQL)

Join notation:

```sql
SELECT ...
FROM A
JOIN_TYPE B
  ON A.key = B.key;
```

In [ ]:
# INNER JOIN and LEFT JOIN (runnable)
inner_q = """
SELECT
    r.ride_date,
    s.station_name,
    s.region,
    r.riders
FROM rides r
INNER JOIN stations s
    ON r.station_id = s.station_id
WHERE s.region = 'Ile-de-France'
ORDER BY r.ride_date, s.station_name
LIMIT 10;
"""

left_q = """
SELECT
    d.ride_date,
    d.region,
    d.riders,
    w.mean_temp_c,
    w.precip_mm
FROM (
    SELECT r.ride_date, s.region, SUM(r.riders) AS riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    GROUP BY r.ride_date, s.region
) d
LEFT JOIN weather w
    ON d.ride_date = w.weather_date
   AND d.region = w.region
ORDER BY d.ride_date, d.region
LIMIT 10;
"""

run_sql(inner_q), run_sql(left_q)

In [ ]:
# RIGHT JOIN / FULL OUTER JOIN in PostgreSQL notation (reference)
postgres_right_join = """
SELECT a.*, b.*
FROM table_a a
RIGHT JOIN table_b b ON a.id = b.id;
"""

postgres_full_join = """
SELECT a.*, b.*
FROM table_a a
FULL OUTER JOIN table_b b ON a.id = b.id;
"""

print("PostgreSQL RIGHT JOIN syntax:", postgres_right_join)
print("PostgreSQL FULL OUTER JOIN syntax:", postgres_full_join)


In [ ]:
# GROUP BY, HAVING, ORDER BY
agg_q = """
SELECT
    s.region,
    s.line_code,
    COUNT(*) AS row_count,
    SUM(r.riders) AS total_riders,
    AVG(r.riders) AS avg_riders
FROM rides r
JOIN stations s ON r.station_id = s.station_id
GROUP BY s.region, s.line_code
HAVING AVG(r.riders) > 1000
ORDER BY total_riders DESC;
"""

agg = run_sql(agg_q)
agg

In [ ]:
# Subquery + CTE + Window function
cte_q = """
WITH daily AS (
    SELECT
        r.ride_date,
        s.region,
        SUM(r.riders) AS total_riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    GROUP BY r.ride_date, s.region
)
SELECT
    ride_date,
    region,
    total_riders,
    AVG(total_riders) OVER (
        PARTITION BY region
        ORDER BY ride_date
        ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
    ) AS rolling_7d_avg,
    ROW_NUMBER() OVER (
        PARTITION BY region
        ORDER BY total_riders DESC
    ) AS demand_rank
FROM daily
ORDER BY ride_date, region
LIMIT 20;
"""

cte_df = run_sql(cte_q)
cte_df.head(10)

In [ ]:
# Transaction control (TCL): BEGIN / COMMIT / ROLLBACK
conn.execute("BEGIN")
conn.execute("INSERT INTO ticket_sales (sale_id, station_id, amount, sale_date, payment_method) VALUES (99, 302, 35.0, '2024-01-05', 'card')")
interim = run_sql("SELECT COUNT(*) AS c FROM ticket_sales")

conn.execute("ROLLBACK")
after_rollback = run_sql("SELECT COUNT(*) AS c FROM ticket_sales")

interim, after_rollback

## PostgreSQL-specific syntax you will use in this project

- Auto id: `BIGSERIAL`
- Type cast: `value::numeric`
- Case-insensitive match: `ILIKE '%metro%'`
- Date extraction: `EXTRACT(DOW FROM demand_date)`
- Upsert: `INSERT ... ON CONFLICT (...) DO UPDATE`
- JSONB operators: `->`, `->>`, `@>`
- Copy ingestion: `COPY table FROM 'file.csv' CSV HEADER`

In [ ]:
# Mini project query: demand + weather
mini_q = """
WITH daily AS (
    SELECT
        r.ride_date,
        s.region,
        SUM(r.riders) AS total_riders
    FROM rides r
    JOIN stations s ON r.station_id = s.station_id
    GROUP BY r.ride_date, s.region
)
SELECT
    d.ride_date,
    d.region,
    d.total_riders,
    w.mean_temp_c,
    w.precip_mm,
    CASE WHEN w.precip_mm >= 5 THEN 1 ELSE 0 END AS heavy_rain_flag
FROM daily d
LEFT JOIN weather w
    ON d.ride_date = w.weather_date
   AND d.region = w.region
ORDER BY d.ride_date, d.region;
"""

mini = run_sql(mini_q)
mini.head()

In [ ]:
# Visualization from SQL output
fig, ax = plt.subplots(figsize=(11, 4))
for region, g in mini.groupby("region"):
    ax.plot(pd.to_datetime(g["ride_date"]), g["total_riders"], label=region)
ax.set_title("SQL output: daily riders by region")
ax.set_xlabel("Date")
ax.set_ylabel("Total riders")
ax.legend()
plt.show()

corr = mini[["total_riders", "mean_temp_c", "precip_mm", "heavy_rain_flag"]].corr(numeric_only=True)
corr

## Final checklist

If you can do the following, you are ready for this project:

- Read and clean transport time-series data
- Explain big-data constraints in pipeline design
- Use SQL DDL/DML/TCL confidently
- Write joins, CTEs, aggregations, and window queries
- Understand PostgreSQL syntax needed for production pipeline